In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
import json

from neo4j import GraphDatabase


Setting up connection to the neo4j graph

In [ ]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

Check to see the graph has been imported properly

In [7]:
# Cypher for the total number of nodes in the database.
cypher = "MATCH (n) RETURN count(n)"

records_total_num_nodes, summary_total_num_nodes, keys_total_num_nodes = driver.execute_query(cypher)

print(records_total_num_nodes)

[<Record count(n)=23300>]


Creating a session to add political positions to the graph.

In [ ]:
data_base_connection = GraphDatabase.driver(uri = "bolt://localhost:7687", auth=("neo4j", "password"))
session = data_base_connection.session()  

Creating the system prompt

In [ ]:
sys_msg = {"role" : "system",
           "content" : """You will be provided with the text of a locution and its corresponding propositional content that forms part of an argument from a UK political debating TV programme. 
           Your task is to decide where does the speaker stand on the 'left' to 'right' wing scale using the speaker's locution and propositional content? 
           Provide your response as a score between 0 and 100 where 0 means 'Extremely left' and 100 means 'Extremely right'. If the text does not have political content, set the score to “NA”. 
           Output in JSON format using the following template: {'Score' : int}. 
           
           Do not include any additional context, preamble, or explanation."""
           }

In [11]:
json_list = []

for i in range(0, records_total_num_nodes[0]["count(n)"]):
    cypher = "MATCH (n) WHERE n.unique_id = '"+str(i)+"' RETURN (n)"
    records, summary, keys = driver.execute_query(cypher)

    human_msg = {"role" : "user",
                 "content" : """Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right'), and 'NA' if there is no political content, using the following locution and proposition.
                 
                 Proposition: '""" + records[0]["n"]._properties["proposition"] + """'

                 Locution: '""" + records[0]["n"]._properties["locution"].split(":", 1)[-1].strip() + """'

                 Do not write an introduction or summary. Output in JSON format using the following template: {'Score' : int}"""}
    

    json_entry = {"id" : str(i), "messages" : [sys_msg, human_msg]}

    json_list.append(json_entry)

BufferError: Existing exports of data: object cannot be re-sized

In [ ]:
json_list[:10]

[{'id': '0',
  'messages': [{'role': 'system',
    'content': "You will be provided with the text of a locution and its corresponding propositional content that forms part of an argument from a UK political debating TV programme. \n           Your task is to decide where does the speaker stand on the ‘left’ to ‘right’ wing scale using the speaker's locution and propositional content? \n           Provide your response as a score between 0 and 100 where 0 means ‘Extremely left’ and 100 means ‘Extremely right’. If the text does not have political content, set the score to “NA”. \n           Output in JSON format using the following template: {'Score' : int}. \n           \n           Do not include any additional context, preamble, or explanation."},
   {'role': 'user',
    'content': "Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right'), and 'NA' if there is no political content, using the following locution and proposition.\n                 \

In [ ]:
with open('prompts.jsonl', 'w') as outfile:
    for entry in json_list:
        json.dump(entry, outfile)
        outfile.write('\n')

In [ ]:
gpt_o1_sys_msg = {"role" : "assistant",
           "content" : """You will be provided with the text of a locution and its corresponding propositional content that forms part of an argument from a UK political debating TV programme. 
           Your task is to decide where does the speaker stand on the 'left' to 'right' wing scale using the speaker's locution and propositional content? 
           Provide your response as a score between 0 and 100 where 0 means 'Extremely left' and 100 means 'Extremely right'. If the text does not have political content, set the score to “NA”. 
           Output in JSON format using the following template: {'Score' : int}. 
           
           Do not include any additional context, preamble, or explanation."""
           }

In [14]:
gpt_o1_json_list = []

for i in range(0, records_total_num_nodes[0]["count(n)"]):
    cypher = "MATCH (n) WHERE n.unique_id = '"+str(i)+"' RETURN (n)"
    records, summary, keys = driver.execute_query(cypher)

    human_msg = {"role" : "user",
                 "content" : """Score the speaker's political position between 0 ('Extremely left') and 100 ('Extremely right'), and 'NA' if there is no political content, using the following locution and proposition.
                 
                 Proposition: '""" + records[0]["n"]._properties["proposition"] + """'

                 Locution: '""" + records[0]["n"]._properties["locution"].split(":", 1)[-1].strip() + """'

                 Do not write an introduction or summary. Output in JSON format using the following template: {'Score' : int}"""}
    

    json_entry = {"id" : str(i), "messages" : [gpt_o1_sys_msg, human_msg]}

    gpt_o1_json_list.append(json_entry)

In [17]:
with open('gpt_01_prompts.jsonl', 'w') as outfile:
    for entry in gpt_o1_json_list:
        json.dump(entry, outfile)
        outfile.write('\n')